# Training wake-word models for Nova (experimental)

This notebook trains two small models with [openWakeWord](https://github.com/dscripka/openWakeWord): one for **"hey nova"** and one for **"nova"**. It is an experiment: the results are scored against Nova's test set before anything is put into the app.

**Before you start**

1. Menu: **Runtime → Change runtime type → T4 GPU → Save.**
2. Run the cells from top to bottom, one at a time. Wait for each to finish, and read its last line.
3. The whole thing takes roughly two to three hours. Keep the tab open.

**What data this uses, and where it goes**

- Everything runs on a temporary Google computer that is erased when the session ends.
- **No recording of your voice is used or uploaded.** The examples of "hey nova" are made by a text-to-speech program (Piper) inside this notebook.
- It downloads public research datasets: room echoes (MIT), background sounds (AudioSet), music (Free Music Archive), and features of about 2,000 hours of general speech (ACAV100M, 17 GB).
- At the end it gives you one small file to download, `nova-wake-models.zip`.

The training tools were written for an older Python than Colab now has, so cell 1 sets up its own Python 3.10 with the versions they expect. Cell 4 does a short trial before the long runs.

In [ ]:
# 1. Set up the tools (about 5 minutes). The last line must say "ALL TOOLS READY".
import os
os.environ["MPLBACKEND"] = "agg"      # the training tools must not try to use Colab's own chart display
!pip install -q uv
!uv venv -q --python 3.10 /content/env
!uv pip install -q --python /content/env/bin/python "numpy==1.26.4" "scipy==1.11.4" "torch==2.1.2" "torchaudio==2.1.2" "piper-phonemize==1.1.0" webrtcvad \
    "audiomentations==0.33.0" "torch-audiomentations==0.11.0" "speechbrain==0.5.14" "huggingface_hub==0.19.4" "torchinfo==1.8.0" "torchmetrics==1.2.0" \
    "mutagen==1.47.0" "acoustics==0.2.6" "pronouncing==0.2.0" "datasets==2.14.6" "pyarrow==14.0.2" "fsspec==2023.9.2" "deep-phonemizer==0.0.19" \
    "onnx==1.14.1" "onnxruntime==1.16.3" "scikit-learn==1.3.2" pyyaml tqdm requests soundfile librosa "setuptools==69.5.1"
# The speech generator, at the version the training script was written for.
!git clone -q -b v2.0.0 https://github.com/rhasspy/piper-sample-generator /content/piper-sample-generator
!wget -q -O /content/piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!git clone -q https://github.com/dscripka/openwakeword /content/openwakeword
!uv pip install -q --python /content/env/bin/python --no-deps -e /content/openwakeword
# The training script tries to make a second file format that needs tools Nova does not use. Switch that off.
!sed -i 's/default="False"/default=False/' /content/openwakeword/openwakeword/train.py
import os
os.makedirs("/content/openwakeword/openwakeword/resources/models", exist_ok=True)
for f in ["embedding_model.onnx", "melspectrogram.onnx"]:
    os.system(f"wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/{f} -O /content/openwakeword/openwakeword/resources/models/{f}")
check = """
import sys
sys.path.insert(0, '/content/piper-sample-generator')
import torch, torchaudio, numpy, scipy, yaml, datasets, pyarrow, speechbrain, audiomentations, torch_audiomentations, acoustics, pronouncing, mutagen, torchinfo, torchmetrics, onnxruntime, piper_phonemize, webrtcvad, dp
import openwakeword
from openwakeword.data import generate_adversarial_texts, augment_clips, mmap_batch_generator
from generate_samples import generate_samples
print('python', sys.version.split()[0], '| torch', torch.__version__, '| GPU:', torch.cuda.is_available())
print('ALL TOOLS READY' if torch.cuda.is_available() else 'TOOLS INSTALLED, BUT NO GPU: set Runtime > Change runtime type > T4 GPU and start again')
"""
open("/content/check.py", "w").write(check)
!/content/env/bin/python /content/check.py 2>&1 | tail -4

In [ ]:
# 2. Download the public datasets (about 15 to 25 minutes; the big one is 17 GB). The last line must say "DOWNLOADS FINISHED".
import os
os.environ["MPLBACKEND"] = "agg"      # the training tools must not try to use Colab's own chart display
script = """
import os, numpy as np, scipy.io.wavfile, datasets
from pathlib import Path
from tqdm import tqdm
os.chdir('/content')
os.makedirs('mit_rirs', exist_ok=True)
for row in tqdm(datasets.load_dataset('davidscripka/MIT_environmental_impulse_responses', split='train', streaming=True)):
    scipy.io.wavfile.write(os.path.join('mit_rirs', row['audio']['path'].split('/')[-1]), 16000, (row['audio']['array'] * 32767).astype(np.int16))
os.makedirs('audioset_16k', exist_ok=True)
# Background sounds: one file of AudioSet, which now comes as a table with the sound stored inside it.
import io, soundfile, librosa, pyarrow.parquet as pq
table = pq.ParquetFile('audioset/09.parquet')
column = [f.name for f in table.schema_arrow if 'audio' in f.name.lower()][0]
made = 0
for batch in table.iter_batches(batch_size=32, columns=[column]):
    for item in batch.column(0).to_pylist():
        try:
            sound, rate = soundfile.read(io.BytesIO(item['bytes']), dtype='float32')
            if sound.ndim > 1: sound = sound.mean(axis=1)
            if rate != 16000: sound = librosa.resample(sound, orig_sr=rate, target_sr=16000)
            scipy.io.wavfile.write(os.path.join('audioset_16k', 'bg_%05d.wav' % made), 16000, (np.clip(sound, -1, 1) * 32767).astype(np.int16))
            made += 1
        except Exception:
            pass
    if made >= 900: break
os.makedirs('fma', exist_ok=True)
try:
    fma = iter(datasets.load_dataset('rudraml/fma', name='small', split='train', streaming=True).cast_column('audio', datasets.Audio(sampling_rate=16000)))
    for i in tqdm(range(120)):
        row = next(fma)
        scipy.io.wavfile.write(os.path.join('fma', row['audio']['path'].split('/')[-1].replace('.mp3', '.wav')), 16000, (row['audio']['array'] * 32767).astype(np.int16))
except Exception as e:
    print('The music set could not be fetched (' + str(e)[:80] + '). Carrying on with the background sounds only.')
print('room echoes:', len(os.listdir('mit_rirs')), '| background sounds:', len(os.listdir('audioset_16k')), '| music:', len(os.listdir('fma')))
"""
open("/content/get_data.py", "w").write(script)
import os
os.makedirs("/content/audioset", exist_ok=True)
!wget -q -O /content/audioset/09.parquet https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train/09.parquet
!/content/env/bin/python /content/get_data.py 2>&1 | tail -6
!cd /content && wget -q --show-progress -nc https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!cd /content && wget -q --show-progress -nc https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
big = os.path.getsize("/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy") / 1e9 if os.path.exists("/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy") else 0
print("DOWNLOADS FINISHED" if big > 17 and len(os.listdir("/content/audioset_16k")) > 100 and len(os.listdir("/content/mit_rirs")) > 100 else f"SOMETHING IS MISSING (big file {big:.1f} GB). Send this cell's output back.")

In [ ]:
# 3. The settings shared by both models. The last line must say "SETTINGS READY".
script = """
import yaml, sys, os, subprocess
os.chdir('/content')
# Words and phrases the models must learn NOT to wake on. These are the ones Nova's tests found troublesome.
NOT_THE_NAME = ['no', 'now', 'know', 'nope', 'no way', 'noah', 'novel', 'november', 'nobody', 'over', 'innovate', 'innovation', 'nevada', 'lava', 'mocha', 'nola',
                'know that', 'no thanks', 'not now', 'hey no', 'hey now', 'hey noah', 'hey you', 'nova scotia', 'no the', 'know the', 'never', 'navy', 'hey nobody', 'hey november']
phrase, name, samples, steps = sys.argv[1], sys.argv[2], int(sys.argv[3]), int(sys.argv[4])
c = yaml.load(open('openwakeword/examples/custom_model.yml').read(), yaml.Loader)
c['target_phrase'] = [phrase]
c['model_name'] = name
c['custom_negative_phrases'] = [p for p in NOT_THE_NAME if p != phrase]
c['n_samples'] = samples
c['n_samples_val'] = max(200, samples // 5)
c['steps'] = steps
c['piper_sample_generator_path'] = '/content/piper-sample-generator'
c['rir_paths'] = ['/content/mit_rirs']
backgrounds = ['/content/audioset_16k'] + (['/content/fma'] if len(os.listdir('/content/fma')) > 10 else [])
c['background_paths'] = backgrounds
c['background_paths_duplication_rate'] = [1] * len(backgrounds)
c['false_positive_validation_data_path'] = '/content/validation_set_features.npy'
c['feature_data_files'] = {'ACAV100M_sample': '/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'}
c['output_dir'] = '/content/models_' + name
yaml.dump(c, open(name + '.yaml', 'w'))
for stage in ['--generate_clips', '--augment_clips', '--train_model']:
    code = subprocess.call([sys.executable, 'openwakeword/openwakeword/train.py', '--training_config', name + '.yaml', stage])
    print('STAGE', stage, 'finished with code', code, flush=True)
    if code != 0: break
out = '/content/models_' + name + '/' + name + '.onnx'
print(('MODEL SAVED: ' if os.path.exists(out) else 'NO MODEL WAS PRODUCED: ') + out)
"""
open("/content/train_one.py", "w").write(script)
print("SETTINGS READY")

In [ ]:
# 4. A short trial with a tiny amount of data (5 to 10 minutes), to make sure every stage works before the long runs.
#    The last line must start with "MODEL SAVED". If it says "NO MODEL WAS PRODUCED", stop and send this cell's output back.
import os
os.environ["MPLBACKEND"] = "agg"      # the training tools must not try to use Colab's own chart display
!/content/env/bin/python /content/train_one.py "hey nova" trial_run 150 400 2>&1 | grep -v "Warning\|warn" | tail -25

In [ ]:
# 5. Train "hey nova" (roughly an hour). The last line must start with "MODEL SAVED".
import os
os.environ["MPLBACKEND"] = "agg"      # the training tools must not try to use Colab's own chart display
!/content/env/bin/python /content/train_one.py "hey nova" hey_nova 6000 25000 2>&1 | grep -v "Warning\|warn" | tail -30

In [ ]:
# 6. Train "nova" on its own (roughly an hour). The last line must start with "MODEL SAVED".
import os
os.environ["MPLBACKEND"] = "agg"      # the training tools must not try to use Colab's own chart display
!/content/env/bin/python /content/train_one.py "nova" nova 6000 25000 2>&1 | grep -v "Warning\|warn" | tail -30

In [ ]:
# 7. Package the two models and download them
import shutil, os
os.makedirs("/content/nova-wake-models", exist_ok=True)
for n in ["hey_nova", "nova"]:
    p = f"/content/models_{n}/{n}.onnx"
    if os.path.exists(p): shutil.copy(p, "/content/nova-wake-models/")
    else: print("missing:", p)
shutil.make_archive("/content/nova-wake-models", "zip", "/content/nova-wake-models")
from google.colab import files
files.download("/content/nova-wake-models.zip")

## Afterwards

Leave `nova-wake-models.zip` in your Downloads folder and say it is there. The models are then scored on Nova's test recordings, and only if the numbers are good is anything tried with a real microphone.

**Licence note.** openWakeWord's code is Apache 2.0. The general-speech features used here come from datasets whose own licences are unclear, which is why openWakeWord publishes its ready-made models as non-commercial, share-alike (CC BY-NC-SA 4.0). Models trained here should be treated the same way.